In [1]:

from datasets import load_dataset

DATASET_NAME = "ai4privacy/pii-masking-200k"

dataset = load_dataset(DATASET_NAME)

print("Available splits:", list(dataset.keys()))

for split_name, split_data in dataset.items():
    print(f"\n--- {split_name.upper()} ---")
    print("Number of rows:", len(split_data))
    print("Columns:", split_data.column_names)
    print("Features:", split_data.features)


d:\confidentiality-detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\confidentiality-detector\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\datasets--ai4privacy--pii-masking-200k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate dev

Available splits: ['train']

--- TRAIN ---
Number of rows: 209261
Columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']
Features: {'source_text': Value('string'), 'target_text': Value('string'), 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}), 'span_labels': Value('string'), 'mbert_text_tokens': List(Value('string')), 'mbert_bio_labels': List(Value('string')), 'id': Value('int64'), 'language': Value('string'), 'set': Value('string')}


In [2]:

import json

train_data = dataset["train"]

for i in range(min(3, len(train_data))):
    row = train_data[i]

    print(f"\n{'=' * 60}")
    print(f"RECORD {i + 1}")
    print(f"Language: {row['language']}")
    print(f"Set: {row['set']}")

    print("\nSource text:")
    print(row["source_text"])

    print("\nTarget text:")
    print(row["target_text"])

    print("\nPrivacy mask:")
    print(json.dumps(
        row["privacy_mask"],
        ensure_ascii=False,
        indent=2
    ))

    print("\nSpan labels:")
    print(row["span_labels"])



RECORD 1
Language: en
Set: train

Source text:
A student's assessment was found on device bearing IMEI: 06-184755-866851-3. The document falls under the various topics discussed in our Optimization curriculum. Can you please collect it?

Target text:
A student's assessment was found on device bearing IMEI: [PHONEIMEI]. The document falls under the various topics discussed in our [JOBAREA] curriculum. Can you please collect it?

Privacy mask:
[
  {
    "value": "06-184755-866851-3",
    "start": 57,
    "end": 75,
    "label": "PHONEIMEI"
  },
  {
    "value": "Optimization",
    "start": 138,
    "end": 150,
    "label": "JOBAREA"
  }
]

Span labels:
[[0, 57, "O"], [57, 75, "PHONEIMEI"], [75, 138, "O"], [138, 150, "JOBAREA"], [150, 189, "O"]]

RECORD 2
Language: en
Set: train

Source text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Target text:
Dear [FIRSTNA

In [4]:


from collections import Counter

# 1. Explore language distribution
language_counts = Counter(dataset["train"]["language"])

print("Language distribution:")
for language, count in language_counts.most_common():
    print(f"{language}: {count:,}")

# 2. Filter English records only
english_data = dataset["train"].filter(
    lambda row: row["language"].strip().lower() == "en"
)

print("\nTotal records:", len(dataset["train"]))
print("English records:", len(english_data))
print(
    "English percentage:",
    round(len(english_data) / len(dataset["train"]) * 100, 2),
    "%"
)


Language distribution:
fr: 61,958
de: 52,817
it: 50,985
en: 43,501


Filter: 100%|██████████| 209261/209261 [00:03<00:00, 64793.15 examples/s]


Total records: 209261
English records: 43501
English percentage: 20.79 %


In [5]:

from collections import Counter

# Count entities and records for each label
entity_counts = Counter()
record_counts = Counter()

for row in english_data:
    labels_in_record = set()

    for entity in row["privacy_mask"]:
        label = entity["label"]

        entity_counts[label] += 1
        labels_in_record.add(label)

    record_counts.update(labels_in_record)

# Create a sorted summary
labels_summary = sorted(
    entity_counts.keys(),
    key=lambda label: entity_counts[label],
    reverse=True
)

print("Number of unique labels:", len(labels_summary))
print("\nLabel Distribution:\n")

print(f"{'Label':<25} {'Entities':>12} {'Records':>12}")
print("-" * 51)

for label in labels_summary:
    print(
        f"{label:<25} "
        f"{entity_counts[label]:>12,} "
        f"{record_counts[label]:>12,}"
    )

print("-" * 51)
print(f"{'TOTAL':<25} {sum(entity_counts.values()):>12,}")


Number of unique labels: 56

Label Distribution:

Label                         Entities      Records
---------------------------------------------------
FIRSTNAME                       13,323       12,896
LASTNAME                         4,817        4,741
DATE                             4,555        4,440
EMAIL                            4,092        4,043
PREFIX                           3,446        3,415
AMOUNT                           3,103        3,091
USERNAME                         3,056        2,984
JOBTITLE                         2,954        2,939
URL                              2,910        2,898
TIME                             2,883        2,853
JOBAREA                          2,882        2,858
MIDDLENAME                       2,881        2,871
ACCOUNTNUMBER                    2,831        2,806
STREET                           2,799        2,787
ACCOUNTNAME                      2,793        2,751
CITY                             2,787        2,718
COUNTY        

In [6]:

import pandas as pd

labels_df = pd.DataFrame([
    {
        "label": label,
        "entities": entity_counts[label],
        "records": record_counts[label],
        "record_coverage_pct": round(
            record_counts[label] / len(english_data) * 100, 2
        )
    }
    for label in entity_counts
])

labels_df = labels_df.sort_values(
    "entities", ascending=False
).reset_index(drop=True)

print("Total unique labels:", len(labels_df))
print(labels_df.to_string(index=False))


Total unique labels: 56
              label  entities  records  record_coverage_pct
          FIRSTNAME     13323    12896                29.65
           LASTNAME      4817     4741                10.90
               DATE      4555     4440                10.21
              EMAIL      4092     4043                 9.29
             PREFIX      3446     3415                 7.85
             AMOUNT      3103     3091                 7.11
           USERNAME      3056     2984                 6.86
           JOBTITLE      2954     2939                 6.76
                URL      2910     2898                 6.66
               TIME      2883     2853                 6.56
            JOBAREA      2882     2858                 6.57
         MIDDLENAME      2881     2871                 6.60
      ACCOUNTNUMBER      2831     2806                 6.45
             STREET      2799     2787                 6.41
        ACCOUNTNAME      2793     2751                 6.32
               C

In [10]:

import pandas as pd
from collections import Counter


target_labels = [
    "PASSWORD",
    "USERNAME",
    "EMAIL",
    "PHONENUMBER",
    "FIRSTNAME",
    "LASTNAME",
    "SSN",
    "CREDITCARDNUMBER",
    "CREDITCARDCVV",
    "IBAN",
    "ACCOUNTNUMBER",
    "STREET",
    "IPV4",
]


entity_counts = Counter()
record_counts = Counter()

for row in english_data:
    labels_in_record = set()

    for entity in row["privacy_mask"]:
        label = entity["label"]

        if label in target_labels:
            entity_counts[label] += 1
            labels_in_record.add(label)

    record_counts.update(labels_in_record)

summary = pd.DataFrame([
    {
        "label": label,
        "entities": entity_counts[label],
        "records": record_counts[label],
        "coverage_pct": round(
            record_counts[label] / len(english_data) * 100, 2
        ),
    }
    for label in target_labels
])

print("Labels not found:",
      [label for label in target_labels if entity_counts[label] == 0])

print("\nTotal target entities:", summary["entities"].sum())
print("Records containing at least one target label:",
      sum(
          1 for row in english_data
          if any(e["label"] in target_labels
                 for e in row["privacy_mask"])
      ))

display(summary)


Labels not found: []

Total target entities: 46146
Records containing at least one target label: 30102


,label,entities,records,coverage_pct
0,PASSWORD,2622,2607,5.99
1,USERNAME,3056,2984,6.86
2,EMAIL,4092,4043,9.29
3,PHONENUMBER,2425,2417,5.56
4,FIRSTNAME,13323,12896,29.65
5,LASTNAME,4817,4741,10.90
6,SSN,2045,2032,4.67
7,CREDITCARDNUMBER,2574,2566,5.90
8,CREDITCARDCVV,869,869,2.00
9,IBAN,1973,1968,4.52


In [11]:

# Keep records containing at least one target entity.
target_set = set(target_labels)

scoped_data = english_data.filter(
    lambda row: any(
        entity["label"] in target_set
        for entity in row["privacy_mask"]
    )
)

print("English records:", len(english_data))
print("Scoped records:", len(scoped_data))
print("Target labels:", len(target_labels))

# Inspect one record
sample = scoped_data[0]

print("\nSource text:")
print(sample["source_text"])

print("\nPrivacy mask:")
print(sample["privacy_mask"])


Filter: 100%|██████████| 43501/43501 [00:02<00:00, 17237.02 examples/s]

English records: 43501
Scoped records: 30102
Target labels: 13

Source text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Privacy mask:
[{'value': 'Omer', 'start': 5, 'end': 9, 'label': 'FIRSTNAME'}, {'value': '78B5R2MVFAHJ48500', 'start': 44, 'end': 61, 'label': 'VEHICLEVIN'}]


In [12]:

target_set = set(target_labels)

records_with_both = 0
records_with_only_target = 0

for row in scoped_data:
    labels = [e["label"] for e in row["privacy_mask"]]

    has_target = any(label in target_set for label in labels)
    has_other = any(label not in target_set for label in labels)

    if has_target and has_other:
        records_with_both += 1
    else:
        records_with_only_target += 1

print("Records with target + other labels:", records_with_both)
print("Records with target labels only:", records_with_only_target)


Records with target + other labels: 27533
Records with target labels only: 2569


In [13]:

sample = scoped_data[0]

print("Source text:")
print(sample["source_text"])

print("\nTokens:")
print(sample["mbert_text_tokens"][:40])

print("\nBIO labels:")
print(sample["mbert_bio_labels"][:40])

print("\nSpan labels:")
print(sample["span_labels"])


Source text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Tokens:
['Dear', 'Omer', ',', 'as', 'per', 'our', 'records', ',', 'your', 'license', '78', '##B', '##5', '##R', '##2', '##M', '##V', '##FA', '##H', '##J', '##48', '##50', '##0', 'is', 'still', 'registered', 'in', 'our', 'records', 'for', 'access', 'to', 'the', 'educational', 'tools', '.', 'Please', 'feedback', 'on', 'it']

BIO labels:
['O', 'B-FIRSTNAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'I-VEHICLEVIN', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Span labels:
[[0, 5, "O"], [5, 9, "FIRSTNAME"], [9, 44, "O"], [44, 61, "VEHICLEVIN"], [61, 170, "O"]]


In [14]:

sample = scoped_data[0]

for entity in sample["privacy_mask"]:
    extracted = sample["source_text"][
        entity["start"]:entity["end"]
    ]

    print("Label:", entity["label"])
    print("Expected:", entity["value"])
    print("Extracted:", extracted)
    print("Match:", extracted == entity["value"])
    print("-" * 30)


Label: FIRSTNAME
Expected: Omer
Extracted: Omer
Match: True
------------------------------
Label: VEHICLEVIN
Expected: 78B5R2MVFAHJ48500
Extracted: 78B5R2MVFAHJ48500
Match: True
------------------------------


In [15]:

from collections import Counter

total_entities = 0
matched_entities = 0
mismatched_entities = 0
mismatch_examples = []

for row_idx, row in enumerate(english_data):
    text = row["source_text"]

    for entity in row["privacy_mask"]:
        total_entities += 1

        start = entity["start"]
        end = entity["end"]
        expected = entity["value"]
        extracted = text[start:end]

        if extracted == expected:
            matched_entities += 1
        else:
            mismatched_entities += 1

            if len(mismatch_examples) < 10:
                mismatch_examples.append({
                    "row_index": row_idx,
                    "label": entity["label"],
                    "expected": expected,
                    "extracted": extracted,
                    "start": start,
                    "end": end,
                })

print("Total entities:", total_entities)
print("Matched entities:", matched_entities)
print("Mismatched entities:", mismatched_entities)

if total_entities:
    print(
        "Match rate:",
        round(matched_entities / total_entities * 100, 2),
        "%"
    )

print("\nMismatch examples:")
for example in mismatch_examples:
    print(example)


Total entities: 137093
Matched entities: 137093
Mismatched entities: 0
Match rate: 100.0 %

Mismatch examples:


In [16]:

target_set = set(target_labels)

target_entity_count = sum(
    1
    for row in english_data
    for entity in row["privacy_mask"]
    if entity["label"] in target_set
)

print("All English entities:", total_entities)
print("Target entities:", target_entity_count)
print("Non-target entities:", total_entities - target_entity_count)


All English entities: 137093
Target entities: 46146
Non-target entities: 90947


In [17]:

# Target labels selected for Confidentiality Detector
target_labels = [
    "PASSWORD",
    "USERNAME",
    "EMAIL",
    "PHONENUMBER",
    "FIRSTNAME",
    "LASTNAME",
    "SSN",
    "CREDITCARDNUMBER",
    "CREDITCARDCVV",
    "IBAN",
    "ACCOUNTNUMBER",
    "STREET",
    "IPV4",
]

target_set = set(target_labels)

def build_target_spans(row):
    text = row["source_text"]
    target_spans = []

    for entity in row["privacy_mask"]:
        if entity["label"] not in target_set:
            continue

        start = entity["start"]
        end = entity["end"]
        value = entity["value"]

        # Verify annotation offsets before keeping the entity
        if text[start:end] != value:
            raise ValueError(
                f"Offset mismatch: {entity['label']} | {value!r}"
            )

        target_spans.append({
            "start": start,
            "end": end,
            "label": entity["label"],
            "value": value,
        })

    return target_spans

# Create a new column without modifying the original annotations
scoped_data = scoped_data.map(
    lambda row: {"target_spans": build_target_spans(row)}
)

print("Scoped records:", len(scoped_data))
print("Target labels:", len(target_labels))
print("Example target spans:")
print(scoped_data[0]["target_spans"])


Map: 100%|██████████| 30102/30102 [00:05<00:00, 5218.98 examples/s]

Scoped records: 30102
Target labels: 13
Example target spans:
[{'start': 5, 'end': 9, 'label': 'FIRSTNAME', 'value': 'Omer'}]


In [18]:

sample = scoped_data[0]

print("Source text:")
print(sample["source_text"])

print("\nOriginal labels:")
print(sorted(set(
    entity["label"] for entity in sample["privacy_mask"]
)))

print("\nTarget labels kept:")
print(sorted(set(
    entity["label"] for entity in sample["target_spans"]
)))

print("\nNon-target labels excluded:")
print(sorted(set(
    entity["label"] for entity in sample["privacy_mask"]
    if entity["label"] not in target_set
)))


Source text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Original labels:
['FIRSTNAME', 'VEHICLEVIN']

Target labels kept:
['FIRSTNAME']

Non-target labels excluded:
['VEHICLEVIN']


In [19]:

print("Number of records:", len(scoped_data))
print("Available columns:", scoped_data.column_names)

print("\nTarget labels:", target_labels)

print("\nSample record:")
print(scoped_data[0]["source_text"])


Number of records: 30102
Available columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set', 'target_spans']

Target labels: ['PASSWORD', 'USERNAME', 'EMAIL', 'PHONENUMBER', 'FIRSTNAME', 'LASTNAME', 'SSN', 'CREDITCARDNUMBER', 'CREDITCARDCVV', 'IBAN', 'ACCOUNTNUMBER', 'STREET', 'IPV4']

Sample record:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.


In [20]:

# Inspect target spans in the first record
sample = scoped_data[0]

print("Text:")
print(sample["source_text"])

print("\nTarget spans:")
for span in sample["target_spans"]:
    print(span)

print("\nValidation:")
for span in sample["target_spans"]:
    extracted_text = sample["source_text"][span["start"]:span["end"]]
    print(
        f"Label: {span['label']} | "
        f"Value: {span['value']} | "
        f"Extracted: {extracted_text} | "
        f"Match: {extracted_text == span['value']}"
    )


Text:
Dear Omer, as per our records, your license 78B5R2MVFAHJ48500 is still registered in our records for access to the educational tools. Please feedback on it's operability.

Target spans:
{'start': 5, 'end': 9, 'label': 'FIRSTNAME', 'value': 'Omer'}

Validation:
Label: FIRSTNAME | Value: Omer | Extracted: Omer | Match: True


In [21]:

from transformers import AutoTokenizer

model_checkpoint = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print("Tokenizer loaded successfully!")
print("Tokenizer:", model_checkpoint)


d:\confidentiality-detector\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer loaded successfully!
Tokenizer: bert-base-multilingual-cased


In [22]:

text = "Dear Omer, please contact omer@example.com"

encoding = tokenizer(
    text,
    return_offsets_mapping=True
)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
offsets = encoding["offset_mapping"]

for token, offset in zip(tokens, offsets):
    start, end = offset
    print(f"Token: {token:20} | Offset: {offset} | Text: {text[start:end]}")


Token: [CLS]                | Offset: (0, 0) | Text: 
Token: Dear                 | Offset: (0, 4) | Text: Dear
Token: Omer                 | Offset: (5, 9) | Text: Omer
Token: ,                    | Offset: (9, 10) | Text: ,
Token: pl                   | Offset: (11, 13) | Text: pl
Token: ##eas                | Offset: (13, 16) | Text: eas
Token: ##e                  | Offset: (16, 17) | Text: e
Token: contact              | Offset: (18, 25) | Text: contact
Token: om                   | Offset: (26, 28) | Text: om
Token: ##er                 | Offset: (28, 30) | Text: er
Token: @                    | Offset: (30, 31) | Text: @
Token: example              | Offset: (31, 38) | Text: example
Token: .                    | Offset: (38, 39) | Text: .
Token: com                  | Offset: (39, 42) | Text: com
Token: [SEP]                | Offset: (0, 0) | Text: 


In [23]:

sample = scoped_data[0]
text = sample["source_text"]

encoding = tokenizer(
    text,
    return_offsets_mapping=True,
    truncation=True,
    max_length=256
)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
offsets = encoding["offset_mapping"]

for token, (start, end) in zip(tokens, offsets):
    if start == end:
        continue

    matched_spans = [
        span for span in sample["target_spans"]
        if start < span["end"] and end > span["start"]
    ]

    print(
        f"{token:20} | {(start, end)} | "
        f"{text[start:end]!r} | "
        f"{[span['label'] for span in matched_spans]}"
    )


Dear                 | (0, 4) | 'Dear' | []
Omer                 | (5, 9) | 'Omer' | ['FIRSTNAME']
,                    | (9, 10) | ',' | []
as                   | (11, 13) | 'as' | []
per                  | (14, 17) | 'per' | []
our                  | (18, 21) | 'our' | []
records              | (22, 29) | 'records' | []
,                    | (29, 30) | ',' | []
your                 | (31, 35) | 'your' | []
license              | (36, 43) | 'license' | []
78                   | (44, 46) | '78' | []
##B                  | (46, 47) | 'B' | []
##5                  | (47, 48) | '5' | []
##R                  | (48, 49) | 'R' | []
##2                  | (49, 50) | '2' | []
##M                  | (50, 51) | 'M' | []
##V                  | (51, 52) | 'V' | []
##FA                 | (52, 54) | 'FA' | []
##H                  | (54, 55) | 'H' | []
##J                  | (55, 56) | 'J' | []
##48                 | (56, 58) | '48' | []
##50                 | (58, 60) | '50' | []
##0               

In [24]:

IGNORE_INDEX = -100

def create_bio_labels(text, privacy_mask, offsets):
    labels = []

    for start, end in offsets:
        # Special tokens مثل [CLS] و[SEP]
        if start == end:
            labels.append(IGNORE_INDEX)
            continue

        overlapping_entities = [
            entity for entity in privacy_mask
            if start < entity["end"] and end > entity["start"]
        ]

        if not overlapping_entities:
            labels.append("O")
            continue

        # إذا تداخل الـ Token مع كيان غير مستهدف، نتجاهله
        non_target_overlap = any(
            entity["label"] not in target_set
            for entity in overlapping_entities
        )

        if non_target_overlap:
            labels.append(IGNORE_INDEX)
            continue

        # الكيان المستهدف الذي يتداخل مع هذا الـ Token
        entity = overlapping_entities[0]
        label = entity["label"]

        if start <= entity["start"] < end:
            labels.append(f"B-{label}")
        else:
            labels.append(f"I-{label}")

    return labels


sample = scoped_data[0]

sample_encoding = tokenizer(
    sample["source_text"],
    return_offsets_mapping=True,
    truncation=True,
    max_length=256
)

sample_labels = create_bio_labels(
    sample["source_text"],
    sample["privacy_mask"],
    sample_encoding["offset_mapping"]
)

sample_tokens = tokenizer.convert_ids_to_tokens(
    sample_encoding["input_ids"]
)

for token, label in zip(sample_tokens, sample_labels):
    print(f"{token:20} | {label}")


[CLS]                | -100
Dear                 | O
Omer                 | B-FIRSTNAME
,                    | O
as                   | O
per                  | O
our                  | O
records              | O
,                    | O
your                 | O
license              | O
78                   | -100
##B                  | -100
##5                  | -100
##R                  | -100
##2                  | -100
##M                  | -100
##V                  | -100
##FA                 | -100
##H                  | -100
##J                  | -100
##48                 | -100
##50                 | -100
##0                  | -100
is                   | O
still                | O
registered           | O
in                   | O
our                  | O
records              | O
for                  | O
access               | O
to                   | O
the                  | O
educational          | O
tools                | O
.                    | O
Please               | 

In [25]:

from collections import Counter

print("Label distribution in the sample:")
for label, count in Counter(sample_labels).items():
    print(f"{label}: {count}")

print("\nNumber of tokens:", len(sample_tokens))
print("Number of labels:", len(sample_labels))
print("Lengths match:", len(sample_tokens) == len(sample_labels))


Label distribution in the sample:
-100: 15
O: 31
B-FIRSTNAME: 1

Number of tokens: 47
Number of labels: 47
Lengths match: True


In [26]:

# Find a record with a target entity split across multiple tokens
found = False

for i in range(min(100, len(scoped_data))):
    row = scoped_data[i]

    encoding = tokenizer(
        row["source_text"],
        return_offsets_mapping=True,
        truncation=True,
        max_length=256
    )

    offsets = encoding["offset_mapping"]

    for entity in row["target_spans"]:
        overlapping_tokens = [
            (start, end)
            for start, end in offsets
            if start < end
            and start < entity["end"]
            and end > entity["start"]
        ]

        if len(overlapping_tokens) > 1:
            print("Record index:", i)
            print("Entity:", entity)
            print("Entity text:", row["source_text"][entity["start"]:entity["end"]])
            print("Overlapping token offsets:", overlapping_tokens)
            found = True
            break

    if found:
        break

if not found:
    print("No multi-token target entity found in the first 100 records.")


Record index: 1
Entity: {'start': 0, 'end': 6, 'label': 'FIRSTNAME', 'value': 'Kattie'}
Entity text: Kattie
Overlapping token offsets: [(0, 3), (3, 6)]


In [27]:

row = scoped_data[1]
text = row["source_text"]

encoding = tokenizer(
    text,
    return_offsets_mapping=True,
    truncation=True,
    max_length=256
)

tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])

labels = create_bio_labels(
    text,
    row["privacy_mask"],
    encoding["offset_mapping"]
)

for token, (start, end), label in zip(
    tokens,
    encoding["offset_mapping"],
    labels
):
    if start < end and start < 6 and end > 0:
        print(f"Token: {token:15} | Text: {text[start:end]!r} | Label: {label}")


Token: Kat             | Text: 'Kat' | Label: B-FIRSTNAME
Token: ##tie           | Text: 'tie' | Label: I-FIRSTNAME


In [28]:

checked_records = 0
length_mismatches = 0
missing_target_labels = 0

for i in range(min(100, len(scoped_data))):
    row = scoped_data[i]

    encoding = tokenizer(
        row["source_text"],
        return_offsets_mapping=True,
        truncation=True,
        max_length=256
    )

    labels = create_bio_labels(
        row["source_text"],
        row["privacy_mask"],
        encoding["offset_mapping"]
    )

    if len(encoding["input_ids"]) != len(labels):
        length_mismatches += 1

    offsets = encoding["offset_mapping"]

    for entity in row["target_spans"]:
        # Is the entire entity represented by the tokenized text?
        covered = any(
            start <= entity["start"] and end >= entity["end"]
            for start, end in offsets
            if start < end
        )

        if not covered:
            # A multi-token entity can still be covered by several tokens.
            covered = (
                any(start == entity["start"] for start, end in offsets)
                and any(end == entity["end"] for start, end in offsets)
            )

        if not covered:
            missing_target_labels += 1

    checked_records += 1

print("Records checked:", checked_records)
print("Token/label length mismatches:", length_mismatches)
print("Target entities not fully covered:", missing_target_labels)


Records checked: 100
Token/label length mismatches: 0
Target entities not fully covered: 0
